In [0]:
import pandas as pd
from datetime import datetime

In [0]:
# Read both CSV files
linkedin_df = pd.read_csv("/Workspace/Users/felooamer@gmail.com/linkedin_jobs.csv")
indeed_df   = pd.read_csv("/Workspace/Users/felooamer@gmail.com/indeed_jobs.csv")

# Add source column to track where each job came from
linkedin_df["source"] = "LinkedIn"
indeed_df["source"]   = "Indeed"

# Merge both dataframes into one
df = pd.concat([linkedin_df, indeed_df], ignore_index=True)

print(f"LinkedIn jobs : {len(linkedin_df):,}")
print(f"Indeed jobs   : {len(indeed_df):,}")
print(f"Total         : {len(df):,}")

LinkedIn jobs : 1,071
Indeed jobs   : 1,532
Total         : 2,603


In [0]:
print("=" * 50)
print("NULL CHECK — BEFORE CLEANING")
print("=" * 50)
for col in df.columns:
    nulls = df[col].isna().sum()
    pct   = (nulls / len(df)) * 100
    print(f"  {col:<20} nulls: {nulls:>5,}  ({pct:.1f}%)")

NULL CHECK — BEFORE CLEANING
  title                nulls:     0  (0.0%)
  company              nulls:    82  (3.2%)
  location             nulls:   801  (30.8%)
  date_posted          nulls:    63  (2.4%)
  job_type             nulls:   486  (18.7%)
  job_url              nulls:     0  (0.0%)
  country              nulls:     0  (0.0%)
  source               nulls:     0  (0.0%)


In [0]:
# Remove duplicate jobs by URL
before = len(df)
df = df.drop_duplicates(subset=["job_url"])
print(f"Removed duplicates : {before - len(df):,}")

# Remove rows missing title or company
df = df.dropna(subset=["title", "company"])

# Strip whitespace from text columns
for col in ["title", "company", "location", "country"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# Standardize date format to YYYY-MM-DD
df["date_posted"] = pd.to_datetime(
    df["date_posted"], errors="coerce"
).dt.strftime("%Y-%m-%d")

# Fill missing dates with today's date
df["date_posted"] = df["date_posted"].fillna(
    datetime.now().strftime("%Y-%m-%d")
)

# Sort by most recent postings first
df = df.sort_values("date_posted", ascending=False).reset_index(drop=True)

print(f"Clean total : {len(df):,}")

Removed duplicates : 0
Clean total : 2,521


In [0]:
# Fill missing values with a default value
df["job_type"] = df["job_type"].fillna("Not Specified")
df["date_posted"] = df["date_posted"].fillna("Unknown")

print("Missing values handled successfully!")

Missing values handled successfully!


In [0]:
print("=" * 50)
print("NULL CHECK — AFTER CLEANING")
print("=" * 50)
for col in df.columns:
    nulls = df[col].isna().sum()
    pct   = (nulls / len(df)) * 100
    flag  = "⚠️" if pct > 20 else "✅"
    print(f"  {flag} {col:<20} nulls: {nulls:>5,}  ({pct:.1f}%)")

print()
print("=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"  Total rows       : {len(df):,}")
print(f"  Total columns    : {len(df.columns)}")
print(f"  Date range       : {df['date_posted'].min()} → {df['date_posted'].max()}")
print(f"  Sources          : {df['source'].value_counts().to_dict()}")
print(f"  Countries        : {df['country'].nunique()} unique")
display(df.head(10))

NULL CHECK — AFTER CLEANING
  ✅ title                nulls:     0  (0.0%)
  ✅ company              nulls:     0  (0.0%)
  ✅ location             nulls:     0  (0.0%)
  ✅ date_posted          nulls:     0  (0.0%)
  ✅ job_type             nulls:     0  (0.0%)
  ✅ job_url              nulls:     0  (0.0%)
  ✅ country              nulls:     0  (0.0%)
  ✅ source               nulls:     0  (0.0%)

SUMMARY
  Total rows       : 2,521
  Total columns    : 8
  Date range       : 2018-11-08 → 2026-05-10
  Sources          : {'Indeed': 1450, 'LinkedIn': 1071}
  Countries        : 5 unique


title,company,location,date_posted,job_type,job_url,country,source
Backend Engineer,Payr,nan,2026-05-10,fulltime,https://www.linkedin.com/jobs/view/4409274915,Egypt,LinkedIn
Cloud Ops Engineer,Dicetek LLC,nan,2026-05-10,contract,https://www.linkedin.com/jobs/view/4411834408,UAE,LinkedIn
Cloud Security Engineer (AWS & Azure),Dicetek LLC,nan,2026-05-10,fulltime,https://www.linkedin.com/jobs/view/4411824884,UAE,LinkedIn
Cybersecurity Risk Analyst,Dicetek LLC,nan,2026-05-10,contract,https://www.linkedin.com/jobs/view/4411828534,UAE,LinkedIn
Network Operations Center Engineer,Hire Rightt - Executive Search & HR Advisory,"Dubai, United Arab Emirates",2026-05-10,fulltime,https://www.linkedin.com/jobs/view/4411363470,UAE,LinkedIn
Infrastructure Specialist (AI Networking),Omnix International,nan,2026-05-10,fulltime,https://www.linkedin.com/jobs/view/4411874966,UAE,LinkedIn
Linux Administrator,Dicetek LLC,nan,2026-05-10,contract,https://www.linkedin.com/jobs/view/4411847055,UAE,LinkedIn
Windows & Linux Engineer,LanceSoft Middle East,"Dubai, United Arab Emirates",2026-05-10,contract,https://www.linkedin.com/jobs/view/4411350576,UAE,LinkedIn
Cloud Platform Engineer,LanceSoft Middle East,"Dubai, United Arab Emirates",2026-05-10,contract,https://www.linkedin.com/jobs/view/4411320613,UAE,LinkedIn
"Senior Product Manager, AI Transformation",Crypto.com,nan,2026-05-10,fulltime,https://www.linkedin.com/jobs/view/4400494807,UAE,LinkedIn


In [0]:
# Replace string "nan" and empty values with a proper default value
text_columns = ["title", "company", "location", "country", "job_type"]

for col in text_columns:
    if col in df.columns:
        df[col] = (
            df[col]
            .replace("nan", pd.NA)
            .replace("", pd.NA)
            .fillna("Not Specified")
        )

print("Text columns cleaned successfully!")

Text columns cleaned successfully!


In [0]:
# Save the cleaned dataframe as a Spark Delta Table for permanent storage
spark.createDataFrame(df).write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("tech_jobs")

print("Delta Table saved successfully!")
print(f"Total rows saved : {len(df):,}")

Delta Table saved successfully!
Total rows saved : 2,521


In [0]:
# Read back from Delta Table to confirm everything saved correctly
result = spark.sql("SELECT COUNT(*) as total FROM tech_jobs").toPandas()
print(f"Rows in Delta Table : {result['total'][0]:,}")

# Show breakdown by source and country
display(spark.sql("""
    SELECT source, country, COUNT(*) as count
    FROM tech_jobs
    GROUP BY source, country
    ORDER BY count DESC
"""))

Rows in Delta Table : 2,521


source,country,count
Indeed,UAE,367
Indeed,Saudi Arabia,346
Indeed,Egypt,343
LinkedIn,UAE,271
Indeed,Qatar,268
LinkedIn,Egypt,260
LinkedIn,Saudi Arabia,243
LinkedIn,Qatar,175
Indeed,Kuwait,126
LinkedIn,Kuwait,122


In [0]:
# Create sent_jobs table to track already sent jobs
spark.sql("""
    CREATE TABLE IF NOT EXISTS sent_jobs (
        job_url STRING,
        sent_at TIMESTAMP
    )
""")
print("sent_jobs table ready ✅")

sent_jobs table ready ✅
